# PCU-HYBRID-REATTACHMENT-001 — Protocol v3

Dual-GPU engineering run. GPU0 executes the corrected matched-graph causal reattachment arm. GPU1 independently replays the exact ranking-only L7/K64 mutation and runs the frozen alpha sweep. Results, CSV, visualizations, and the engineering decision are validated and pushed back to the source branch. Formal seeds remain untouched.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-hybrid-reattachment-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-hybrid-reattachment-001/engineering/26090501-l7-k64-ranking-causal-reattach-v3'
WORKERS = Path('/kaggle/working/pcu-hybrid-reattachment-001-v3-workers')
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, f'Need 2 GPUs, found {torch.cuda.device_count()}'
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'gpu1': torch.cuda.get_device_name(1),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
payload = json.loads(SEED_REGISTRY.read_text())
states = {int(row['seed']): row['state'] for row in payload['seeds']}
assert states == {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
assert run(['git', 'status', '--porcelain'], capture=True) == ''
print(json.dumps({'formal_seed_states': states}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/run_pcu_hybrid_reattachment_001.py', '--worker-root', WORKERS, '--out', OUT])
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
print(json.dumps({
    'status': decision['status'],
    'same_graph_zero_state_equivalence_passes': decision['same_graph_zero_state_equivalence_passes'],
    'causal_hybrid_consumption_passes': decision['causal_hybrid_consumption_passes'],
    'alpha1_locality_passes': decision['alpha1_locality_passes'],
    'amplitude_sweep_locality_rescue_passes': decision['amplitude_sweep_locality_rescue_passes'],
    'selected_locality_compatible_point': decision['selected_locality_compatible_point'],
    'primary_causal_effect': result['primary_causal_effect'],
}, indent=2))


In [ ]:
from IPython.display import Image, display
for name in ['equivalence_diffs.png', 'causal_ranking_on_off.png', 'alpha_sweep_tradeoff.png', 'association_locality_pareto.png']:
    print(name)
    display(Image(filename=str(OUT / name)))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_hybrid_reattachment_001.py', '--branch', BRANCH])
print(json.dumps({
    'published_commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'branch': run(['git', 'branch', '--show-current'], capture=True),
    'formal_seed_states': {int(row['seed']): row['state'] for row in json.loads(SEED_REGISTRY.read_text())['seeds']},
}, indent=2))
